# Tratamento de Dados
Este notebook aplica as mesmas transformações do `tratativa.ipynb`, mas opera apenas em DataFrames na memória.
As funções não precisam de caminhos de arquivo e são ideais para uso no Google Colab.

In [ ]:
import pandas as pd
import re

# As funções abaixo operam diretamente sobre DataFrames em memória.
# Ideal para uso em Google Colab sem referências a arquivos locais.

In [ ]:
def limpar_avaliacoes(texto):
    if not isinstance(texto, str):
        return texto

    texto = texto.strip()
    texto = re.sub(r'^[\W_]+', '', texto, flags=re.UNICODE)
    texto = re.sub(r'\s{2,}', ' ', texto)

    texto = re.sub(r'\s+([.,;:!?])', r'\1', texto)
    texto = re.sub(r'([.,;:!?])\1+', r'\1', texto)
    texto = re.sub(r'([.,;:!?])(?=[^\s])', r'\1 ', texto)

    mapeamento = {
        r'\b(n[ãa]o|nã|nao)\b': 'não',
        r'\bvc\b': 'você',
        r'\bvcs\b': 'vocês',
        r'\b(mt|mto)\b': 'muito'
    }

    def aplicar_mapeamento(match):
        original = match.group(0)
        alvo = ''
        for padrao, subst in mapeamento.items():
            if re.search(padrao, original, flags=re.IGNORECASE):
                alvo = subst
                break

        if original.isupper():
            return alvo.upper()
        if original and original[0].isupper():
            return alvo.capitalize()
        return alvo

    regex_completo = '|'.join(mapeamento.keys())
    texto = re.sub(regex_completo, aplicar_mapeamento, texto, flags=re.IGNORECASE)

    texto = re.sub(r'(\.\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), texto)
    texto = re.sub(r'[\s]+$', '', texto)
    return texto

In [ ]:
def juntar_titulo_mensagem(df, coluna_titulo='review_comment_title', coluna_mensagem='review_comment_message'):
    if coluna_titulo not in df.columns or coluna_mensagem not in df.columns:
        print(f'Erro: colunas {coluna_titulo} ou {coluna_mensagem} não encontradas.')
        return df

    df[coluna_titulo] = df[coluna_titulo].astype('string')
    df[coluna_mensagem] = df[coluna_mensagem].astype('string')

    def combinar(titulo, mensagem):
        titulo = str(titulo).strip() if pd.notna(titulo) else ''
        mensagem = str(mensagem).strip() if pd.notna(mensagem) else ''

        if titulo and mensagem:
            return f'{titulo} - {mensagem}'
        return titulo or mensagem

    df[coluna_mensagem] = df.apply(lambda row: combinar(row[coluna_titulo], row[coluna_mensagem]), axis=1)
    return df

In [ ]:
def processar_dataframe(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f'Erro: A coluna {nome_coluna} não existe no DataFrame.')
        return df

    df = df.copy()
    df[nome_coluna] = df[nome_coluna].astype('string').apply(limpar_avaliacoes)
    return df

In [ ]:
def remover_linhas_sem_review(df, coluna='review_comment_message'):
    if coluna not in df.columns:
        print(f'Erro: A coluna {coluna} não existe no DataFrame.')
        return df

    df = df.copy()
    mask = df[coluna].astype('string').str.strip() == ''
    removidas = int(mask.sum())
    df = df.drop(df[mask].index).reset_index(drop=True)
    print(f'Removidas {removidas} linhas sem valor em {coluna}.')
    return df

In [ ]:
def apagar_coluna(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f'Erro: A coluna {nome_coluna} não existe no DataFrame.')
        return df

    df = df.copy()
    df = df.drop(columns=[nome_coluna])
    return df

In [ ]:
def converter_para_string(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
        return df

    df = df.copy()
    df[nome_coluna] = df[nome_coluna].astype('string')
    print(f"Coluna '{nome_coluna}' convertida para string.")
    return df


def converter_para_datetime(df, nome_coluna):
    if nome_coluna not in df.columns:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
        return df

    df = df.copy()
    df[nome_coluna] = pd.to_datetime(df[nome_coluna], errors='coerce')
    print(f"Coluna '{nome_coluna}' convertida para datetime.")
    return df

In [ ]:
# Exemplo de uso em memória no Colab:
df = pd.read_csv('/content/avaliacoes.csv')  # ajuste o caminho conforme o arquivo no Colab

df = processar_dataframe(df, 'review_comment_title')
df = processar_dataframe(df, 'review_comment_message')
df = juntar_titulo_mensagem(df)
df = remover_linhas_sem_review(df, 'review_comment_message')

df = df.reset_index(drop=True)

# Exibir o DataFrame tratado sem gravar o arquivo original
print(df.head())